<div id="singlestore-header" style="display: flex; background-color: rgba(235, 249, 245, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/database.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Build an Application-Ready Marketing Transactions Schema with Sequences</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This notebook requires a Standard or Premium workspace. Sequences are not available on Free Starter Workspaces.</p>
    </div>
</div>

## The application scenario

A subscriber just made a purchase through the Real-Time Digital Marketing platform. Your application needs to:

1. Record the purchase as a **marketing transaction** with a compact, unique identifier
2. Create a linked **order record** and derive a human-readable **invoice display number**
3. Attach one or more **line items** to that order
4. Do all of this safely under concurrent writes — without using `MAX(id) + 1`

By the end of this notebook your application tables will produce output like this:

<table>
  <tr><th>txn_id</th><th>display_ref</th><th>invoice_display</th><th>advertiser</th><th>vendor</th><th>bid_cents</th></tr>
  <tr><td>1</td><td>TXN-00001</td><td>INV-000001</td><td>Quatz</td><td>quatz.biz</td><td>10</td></tr>
  <tr><td>2</td><td>TXN-00002</td><td>INV-000002</td><td>Topiclounge</td><td>topiclounge.com</td><td>10</td></tr>
</table>

> **Note:** Sequence values are database-generated. Your actual `txn_id` and `order_id` values will differ from the table above — this is expected and correct.

**Technical reference:** [A Developer's Guide to SingleStore Sequences](https://www.singlestore.com/blog/guide-to-singlestore-sequences/)

**Dataset:** Real-Time Digital Marketing — `purchases`, `offers`, and `vendors` tables.

**Prerequisites:** Standard or Premium workspace; no external credentials required.

## How the pieces fit together

```
RTDM purchase event  (subscriber_id, vendor_name, ts)
          │
          ▼
  marketing_txns      txn_id  ← sequence-generated
          │
          ▼
    mktx_orders       order_id ← sequence-generated
                      invoice_display = CONCAT('INV-', LPAD(order_id, 6, '0'))
          │
          ▼
  txn_line_items      line_id  ← sequence-generated
```

Each arrow is a one-to-one or one-to-many relationship. The source event columns (`subscriber_id`, `event_ts`, `offer_id`) travel with the transaction row for lineage. The sequence columns are generated by the database — the application never computes them.

## Section 1 — Environment setup

The cells below install dependencies, connect to SingleStore, and create a scoped database. This is environment preparation — the application schema design begins in Section 3.

In [1]:
%pip install singlestoredb pandas --quiet

In [2]:
import random
import time
import pandas as pd
import singlestoredb as s2

pd.set_option("display.max_colwidth", 120)
random.seed(42)

DB_NAME = "mktx_sequences_demo"
USE_DB = None

def connect():
    global conn
    conn = s2.connect(database=USE_DB) if USE_DB else s2.connect()
    conn.autocommit(True)
    if USE_DB:
        with conn.cursor() as cur:
            cur.execute(f"USE {USE_DB}")

def _query(query, params=None):
    with conn.cursor() as cur:
        cur.execute(query, params)
        if cur.description:
            cols = [d[0] for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)
    return None

def run_sql(query, params=None):
    for attempt in range(4):
        try:
            return _query(query, params)
        except Exception as e:
            msg = str(e)
            if getattr(e, "errno", None) in (2006, 2013) or "Lost connection" in msg or "gone away" in msg:
                connect()
            else:
                raise

def try_sql(query, params=None, label="Step"):
    try:
        return run_sql(query, params)
    except Exception as e:
        print(f"{label} skipped: {e}")
        return None

connect()

shared = run_sql("SHOW VARIABLES LIKE 'is_shared_tier'")
IS_SHARED = shared is not None and len(shared) > 0 and shared.iloc[0, 1] == "ON"

if IS_SHARED:
    print("Shared tier detected — notebook will use the current database.")
    USE_DB = run_sql("SELECT DATABASE() AS db").iloc[0, 0]
else:
    run_sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
    USE_DB = DB_NAME
    run_sql(f"USE {DB_NAME}")

connect()
print("Database:", run_sql("SELECT DATABASE() AS db").iloc[0, 0])
print("SingleStore version:", run_sql("SELECT @@memsql_version AS v").iloc[0, 0])

## Section 2 — Inspect the Real-Time Digital Marketing purchase data

Load a bounded, deterministic sample of the RTDM `purchases`, `offers`, and `vendors` tables. The `martech` database is created by the Real-Time Digital Marketing demo; if it is absent, a small sample is streamed from S3 directly.

### Create the RTDM source tables

Drop and recreate the three source tables scoped to this notebook. This block is safe to rerun — `DROP TABLE IF EXISTS` ensures a clean slate.

In [3]:
# Source tables — scoped to this notebook's database
for t in ["rtdm_purchases", "rtdm_offers", "rtdm_vendors"]:
    run_sql(f"DROP TABLE IF EXISTS {t}")

run_sql('''
    CREATE TABLE rtdm_purchases (
        subscriber_id BIGINT,
        vendor_name   VARCHAR(100),
        ts            DATETIME(6) DEFAULT NOW(6),
        SORT KEY (ts),
        KEY (vendor_name) USING HASH
    )
''')

run_sql('''
    CREATE ROWSTORE TABLE rtdm_offers (
        offer_id             BIGINT PRIMARY KEY,
        customer             VARCHAR(100),
        enabled              BOOL,
        notification_zone    TEXT,
        segment_ids          JSON,
        notification_content TEXT,
        notification_target  VARCHAR(200),
        maximum_bid_cents    BIGINT
    )
''')

run_sql('''
    CREATE ROWSTORE REFERENCE TABLE rtdm_vendors (
        domain     VARCHAR(200) PRIMARY KEY,
        vendor     VARCHAR(100),
        category   VARCHAR(50),
        popularity DOUBLE
    )
''')
print("Source tables created.")

### Load RTDM data

Use the live `martech` database if it exists, otherwise load a capped sample from S3.

**Canonical vendor enrichment — what it is and what it is not:**

The `rtdm_purchases` table stores `vendor_name` (a plain-text name decoded from base64 in the S3 source). It does not store a verified campaign domain or a confirmed purchase-to-offer relationship.

The vendors reference data contains one row per *domain* (e.g. `dabshots.mil`, `dabshots.edu`) with the same vendor *name* appearing across multiple TLDs. Joining purchases directly to vendors by name would fan out by up to 9× and inflate row counts. To prevent this, a `rtdm_vendor_canonical` view resolves each vendor name to exactly one domain using `MIN(domain)` as a deterministic tiebreaker. The `category` value comes from that same winning row.

**This is demo enrichment, not attribution:** `MIN(domain)` picks a domain that happens to sort first alphabetically — it does not represent a confirmed attribution between the subscriber's purchase event and that domain's advertising campaign. The 1:1 join only proves that canonicalization prevents row fanout; it does not prove a real campaign relationship.

In [4]:
import json
import base64, io, re, urllib.request

DEMO_DB   = "martech"
MAX_ROWS  = 10_000
BUCKET    = "https://singlestore-realtime-digital-marketing.s3.amazonaws.com"

# --- vendors ---
vdf = pd.read_json(
    "https://raw.githubusercontent.com/singlestore-labs/demo-realtime-digital-marketing/main/data/vendors.json"
)
vdf["domain"] = (vdf["vendor"].str.lower() + "." + vdf["tld"])
vdf = vdf.drop_duplicates("domain")
with conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO rtdm_vendors VALUES (%s, %s, %s, %s)",
        vdf[["domain", "vendor", "category", "popularity"]].values.tolist(),
    )
print(f"Vendors loaded: {len(vdf)} rows ({vdf['vendor'].nunique()} unique names)")

# One canonical domain per vendor name.
# rtdm_purchases stores vendor_name (decoded from base64), not a verified
# source domain, so this mapping is demo enrichment — not confirmed attribution.
# MIN(domain) is the tiebreaker; category comes from that same winning row.
run_sql("DROP VIEW IF EXISTS rtdm_vendor_canonical")
run_sql("""
    CREATE VIEW rtdm_vendor_canonical AS
    SELECT v.vendor, v.domain, v.category
    FROM   rtdm_vendors v
    JOIN   (SELECT vendor, MIN(domain) AS domain FROM rtdm_vendors GROUP BY vendor) m
           ON m.vendor = v.vendor AND m.domain = v.domain
""")
print("Canonical vendor view created.")

# --- purchases + offers ---
demo_ok = False
try:
    n = run_sql(
        "SELECT COUNT(*) AS n FROM information_schema.TABLES "
        "WHERE TABLE_SCHEMA = %s AND TABLE_NAME = 'purchases'", (DEMO_DB,)
    ).iloc[0, 0]
    demo_ok = int(n) > 0
except Exception:
    pass

if demo_ok:
    # martech purchases.vendor stores the plain vendor NAME.
    run_sql(f"""
        INSERT INTO rtdm_purchases (subscriber_id, vendor_name, ts)
        SELECT subscriber_id, vendor, ts
        FROM {DEMO_DB}.purchases
        ORDER BY ts DESC
        LIMIT {MAX_ROWS}
    """)
    run_sql(f"""
        INSERT IGNORE INTO rtdm_offers
        SELECT offer_id, customer, enabled, notification_zone,
               segment_ids, notification_content, notification_target, maximum_bid_cents
        FROM {DEMO_DB}.offers
        ORDER BY offer_id
        LIMIT 500
    """)
    print("Loaded from live martech database.")
else:
    # S3 NDJSON: vendor field is base64-encoded vendor name.
    print("martech DB not found — loading sample purchases from S3...")
    SCALE = "1k-2p"
    listing = urllib.request.urlopen(
        f"{BUCKET}/?list-type=2&prefix=v2/{SCALE}/purchases.", timeout=15
    ).read().decode()
    keys = re.findall(r"<Key>(.*?)</Key>", listing)[:3]
    rows_loaded = 0
    for key in keys:
        raw   = urllib.request.urlopen(f"{BUCKET}/{key}", timeout=20).read().decode()
        lines = [json.loads(l) for l in raw.strip().split("\n") if l.strip()]
        batch = []
        for row in lines:
            name = base64.b64decode(row["vendor"] + "==").decode("utf-8")
            batch.append((int(row["subscriberid"]), name))
        batch = batch[:MAX_ROWS - rows_loaded]
        with conn.cursor() as cur:
            cur.executemany(
                "INSERT INTO rtdm_purchases (subscriber_id, vendor_name) VALUES (%s, %s)",
                batch,
            )
        rows_loaded += len(batch)
        if rows_loaded >= MAX_ROWS:
            break
    # Synthesise one offer per canonical vendor domain
    run_sql("""
        INSERT INTO rtdm_offers
            (offer_id, customer, enabled, notification_zone, segment_ids,
             notification_content, notification_target, maximum_bid_cents)
        SELECT ROW_NUMBER() OVER (ORDER BY domain),
               vendor, TRUE, NULL, '[]',
               CONCAT('Deal at ', vendor), domain, 10
        FROM rtdm_vendor_canonical
    """)
    print(f"Loaded {rows_loaded} purchase rows from S3.")

run_sql('''
    SELECT
        (SELECT COUNT(*) FROM rtdm_purchases) AS purchases,
        (SELECT COUNT(*) FROM rtdm_offers)    AS offers,
        (SELECT COUNT(*) FROM rtdm_vendors)   AS vendors
''')

### Inspect a sample

Confirm the source columns that the transaction layer will consume.

In [5]:
print("--- purchases (source event: decoded vendor name) ---")
display(run_sql(
    "SELECT subscriber_id, vendor_name, ts FROM rtdm_purchases ORDER BY ts DESC LIMIT 5"
))

print("\n--- canonical vendor enrichment (one domain per vendor name, demo only) ---")
display(run_sql(
    "SELECT vendor, domain, category FROM rtdm_vendor_canonical ORDER BY vendor LIMIT 5"
))

print("\n--- row-count check: purchases after canonical enrichment join ---")
display(run_sql('''
    SELECT
        COUNT(*)                                AS purchase_rows,
        COUNT(DISTINCT p.subscriber_id)         AS unique_subscribers,
        COUNT(DISTINCT vc.domain)               AS distinct_enriched_domains,
        COUNT(DISTINCT o.offer_id)              AS distinct_offers
    FROM      rtdm_purchases       p
    JOIN      rtdm_vendor_canonical vc ON vc.vendor = p.vendor_name
    JOIN      rtdm_offers           o  ON o.notification_target = vc.domain
'''))
# purchase_rows == unique_subscribers when each subscriber appears once.
# distinct_enriched_domains == distinct offers confirms 1:1 canonical match,
# not a verified campaign attribution.

print("\n--- offers (synthesised from canonical domains when martech DB absent) ---")
run_sql(
    "SELECT offer_id, customer, notification_target AS domain, maximum_bid_cents "
    "FROM rtdm_offers ORDER BY offer_id LIMIT 5"
)

## Section 3 — Why `MAX(id) + 1` is unsafe

A common mistake is to compute the next identifier in application code:

```python
# UNSAFE — two concurrent writers can read the same MAX and produce duplicate ids
next_id = run_sql("SELECT MAX(txn_id) + 1 AS n FROM marketing_txns").iloc[0, 0]
run_sql(f"INSERT INTO marketing_txns (txn_id, ...) VALUES ({next_id}, ...)")
```

In a distributed system like SingleStore — or even a single node under concurrent writes — two sessions can read the same `MAX` value simultaneously and both try to insert the same id, causing a duplicate-key error or silent data corruption.

`AUTO_INCREMENT AS SEQUENCE` delegates identifier generation to the database. Each aggregator pre-caches a range of values (`sequence_min_cache_count`, default 100), so inserts are fast and unique without table-level locks.

## Section 4 — Create application-owned tables with sequences

Three tables form the application boundary. Each uses `INT AUTO_INCREMENT AS SEQUENCE` as its primary key — the database generates these identifiers, the application never supplies them.

<table>
  <tr><th>Table</th><th>Purpose</th><th>Sequence column</th></tr>
  <tr><td><code>marketing_txns</code></td><td>One row per accepted purchase event</td><td><code>txn_id</code></td></tr>
  <tr><td><code>mktx_orders</code></td><td>One order per transaction; source of the invoice display number</td><td><code>order_id</code></td></tr>
  <tr><td><code>txn_line_items</code></td><td>One row per vendor line within a transaction</td><td><code>line_id</code></td></tr>
</table>

> **SingleStore constraint:** A sequence PK is incompatible with a separate composite unique key. Idempotency is handled at the application layer using `TRUNCATE` before each full load — appropriate for a demo notebook where the full dataset is always reloaded from scratch.

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-info-circle"></b>
    <div>
        <p><b>One sequence per table</b></p>
        <p>SingleStore supports exactly <strong>one</strong> <code>AUTO_INCREMENT AS SEQUENCE</code> column per table. Do not add a second sequence column to the same table.</p>
    </div>
</div>

In [6]:
for t in ["txn_line_items", "mktx_orders", "marketing_txns"]:
    run_sql(f"DROP TABLE IF EXISTS {t}")

# ROWSTORE for the parent table (transactional write pattern).
# Note: SingleStore requires every unique key to include all PK columns.
# With a sequence PK (txn_id), a separate natural-key unique constraint is
# not supported. Idempotency is handled at the application layer (see Section 5).
run_sql('''
    CREATE ROWSTORE TABLE marketing_txns (
        txn_id          INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
        subscriber_id   BIGINT       NOT NULL,
        event_ts        DATETIME(6)  NOT NULL,
        offer_id        BIGINT       NOT NULL,
        customer        VARCHAR(100),
        domain          VARCHAR(200),
        category        VARCHAR(50),
        bid_cents       BIGINT,
        created_at      DATETIME(6) DEFAULT NOW(6),
        KEY idx_src (subscriber_id, event_ts, offer_id) USING HASH
    )
''')

# Order / invoice record — one per transaction
run_sql('''
    CREATE ROWSTORE TABLE mktx_orders (
        order_id    INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
        txn_id      INT  NOT NULL,
        customer    VARCHAR(100),
        domain      VARCHAR(200),
        bid_cents   BIGINT,
        created_at  DATETIME(6) DEFAULT NOW(6),
        KEY idx_txn (txn_id) USING HASH
    )
''')

# Child: optional line items within a transaction
run_sql('''
    CREATE ROWSTORE TABLE txn_line_items (
        line_id     INT  AUTO_INCREMENT AS SEQUENCE PRIMARY KEY,
        txn_id      INT  NOT NULL,
        vendor      VARCHAR(100),
        category    VARCHAR(50),
        bid_cents   BIGINT,
        KEY idx_txn (txn_id) USING HASH
    )
''')
print("Application tables created.")

### Verify the sequence columns

`SHOW CREATE TABLE` confirms that `txn_id` uses `AUTO_INCREMENT AS SEQUENCE`.

In [7]:
run_sql("SHOW CREATE TABLE marketing_txns")

## Section 5 — Insert an order and capture its generated ID

This section is the core payoff. Steps:

1. Insert 200 enriched purchase events into `marketing_txns` — **no `txn_id` supplied**
2. Immediately create one `mktx_orders` row per transaction
3. Use `LAST_INSERT_ID()` to insert a linked `txn_line_items` row for a single event, demonstrating parent-child linking without `MAX(id)`

The tables are truncated first to keep the cell rerun-safe.

In [8]:
# Truncate first so this cell is safe to re-run.
# (A composite unique key on the source event is not supported when the PK
#  is a sequence column — see the note above.)
run_sql("TRUNCATE TABLE txn_line_items")
run_sql("TRUNCATE TABLE mktx_orders")
run_sql("TRUNCATE TABLE marketing_txns")

# Load 200 enriched conversion events via canonical vendor enrichment.
run_sql('''
    INSERT INTO marketing_txns
        (subscriber_id, event_ts, offer_id, customer, domain, category, bid_cents)
    SELECT
        p.subscriber_id,
        p.ts,
        o.offer_id,
        o.customer,
        vc.domain,
        COALESCE(vc.category, 'general'),
        o.maximum_bid_cents
    FROM       rtdm_purchases        p
    JOIN       rtdm_vendor_canonical vc ON vc.vendor              = p.vendor_name
    JOIN       rtdm_offers           o  ON o.notification_target  = vc.domain
    ORDER BY   p.ts DESC
    LIMIT 200
''')

# Create an order record for each transaction (one-to-one)
run_sql('''
    INSERT INTO mktx_orders (txn_id, customer, domain, bid_cents)
    SELECT txn_id, customer, domain, bid_cents
    FROM   marketing_txns
''')

run_sql('''
    SELECT
        (SELECT COUNT(*) FROM marketing_txns) AS txns_loaded,
        (SELECT COUNT(*) FROM mktx_orders)   AS orders_created
''')

### Inspect the generated identifiers

The source event columns (`subscriber_id`, `event_ts`, `offer_id`) travel alongside the database-generated `txn_id`. These serve different purposes:

<table>
  <tr><th>Column</th><th>Purpose</th></tr>
  <tr><td><code>txn_id</code></td><td>Compact application identifier — generated by the sequence, never supplied by the caller</td></tr>
  <tr><td><code>subscriber_id</code>, <code>event_ts</code>, <code>offer_id</code></td><td>Source identity — used for lineage and duplicate detection</td></tr>
</table>

In [9]:
run_sql('''
    SELECT txn_id, subscriber_id, event_ts,
           offer_id, customer, category, bid_cents
    FROM   marketing_txns
    ORDER  BY txn_id
    LIMIT  10
''')

### Parent-child linking with `LAST_INSERT_ID()`

After inserting the parent transaction row, call `LAST_INSERT_ID()` to read back the sequence value the database generated **for this session**. Use that value to insert the child line-item row.

This avoids a race-prone `MAX(txn_id)` lookup — two concurrent sessions each get their own `LAST_INSERT_ID()` result.

In [10]:
# Insert one conversion event and link its order and line items via LAST_INSERT_ID()
sample = run_sql('''
    SELECT p.subscriber_id,
           p.ts          AS event_ts,
           o.offer_id, o.customer,
           vc.domain,
           COALESCE(vc.category, 'general') AS category,
           o.maximum_bid_cents
    FROM      rtdm_purchases        p
    JOIN      rtdm_vendor_canonical vc ON vc.vendor             = p.vendor_name
    JOIN      rtdm_offers           o  ON o.notification_target = vc.domain
    ORDER BY  p.ts
    LIMIT 1
''')

row = sample.iloc[0]

with conn.cursor() as cur:
    # Insert the parent transaction — no txn_id supplied
    cur.execute('''
        INSERT INTO marketing_txns
            (subscriber_id, event_ts, offer_id, customer, domain, category, bid_cents)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    ''', (int(row.subscriber_id), row.event_ts, int(row.offer_id),
           row.customer, row.domain, row.category, int(row.maximum_bid_cents)))

    # Retrieve the sequence value generated for this session
    cur.execute('SELECT LAST_INSERT_ID() INTO @parent_txn_id')

    cur.execute('''
        INSERT INTO mktx_orders (txn_id, customer, domain, bid_cents)
        VALUES (@parent_txn_id, %s, %s, %s)
    ''', (row.customer, row.domain, int(row.maximum_bid_cents)))

    cur.execute('''
        INSERT INTO txn_line_items (txn_id, vendor, category, bid_cents)
        VALUES (@parent_txn_id, %s, %s, %s)
    ''', (row.domain, row.category, int(row.maximum_bid_cents)))

run_sql('''
    SELECT t.txn_id,
           CONCAT('INV-', LPAD(ord.order_id, 6, '0')) AS invoice_ref,
           t.subscriber_id, t.customer, t.category,
           l.line_id, l.vendor, l.bid_cents
    FROM   marketing_txns  t
    JOIN   mktx_orders     ord ON ord.txn_id = t.txn_id
    JOIN   txn_line_items  l   ON l.txn_id   = t.txn_id
    ORDER  BY t.txn_id DESC
    LIMIT  5
''')

### Checkpoint: what should happen when the same purchase event is loaded twice?

Before running the duplicate-event cell below, think about what the application should do:

- Should it create a second transaction with a new `txn_id`?
- Should it silently skip the event?
- Should it raise an error?

The answer depends on whether you look up the source event key `(subscriber_id, event_ts, offer_id)` before inserting. Run the next cell to see the guarded pattern in action.

In [11]:
# Attempt to insert the same purchase event a second time.
# The guard checks whether a transaction already exists for this source event
# before inserting, so no duplicate application record is created.
sample = run_sql('''
    SELECT subscriber_id, event_ts, offer_id
    FROM   marketing_txns
    ORDER  BY txn_id
    LIMIT  1
''')
row = sample.iloc[0]

existing = run_sql('''
    SELECT txn_id FROM marketing_txns
    WHERE subscriber_id = %s AND event_ts = %s AND offer_id = %s
''', (int(row.subscriber_id), row.event_ts, int(row.offer_id)))

if len(existing) > 0:
    print(f"Source event already recorded as txn_id={existing.iloc[0,0]} — skipping insert.")
    print("The source event key (subscriber_id, event_ts, offer_id) is the idempotency key.")
    print("The sequence-generated txn_id is the application key — these are separate concerns.")
else:
    run_sql('''
        INSERT INTO marketing_txns
            (subscriber_id, event_ts, offer_id, customer, domain, category, bid_cents)
        VALUES (%s, %s, %s, '', '', '', 0)
    ''', (int(row.subscriber_id), row.event_ts, int(row.offer_id)))
    print("Inserted (event was new).")

### Validate the generated identifiers

Confirm that every sequence-generated key is unique and that no line items are orphaned.

In [12]:
checks = {}

# Primary key uniqueness
dup_pk = run_sql('''
    SELECT txn_id, COUNT(*) AS cnt
    FROM   marketing_txns
    GROUP  BY txn_id
    HAVING cnt > 1
''')
checks["duplicate txn_id"] = len(dup_pk)

# Order uniqueness (one order per transaction)
dup_ord = run_sql('''
    SELECT order_id, COUNT(*) AS cnt
    FROM   mktx_orders
    GROUP  BY order_id
    HAVING cnt > 1
''')
checks["duplicate order_id"] = len(dup_ord)

# Orphaned line items (no parent txn)
orphans = run_sql('''
    SELECT COUNT(*) AS n
    FROM   txn_line_items l
    LEFT JOIN marketing_txns t ON t.txn_id = l.txn_id
    WHERE  t.txn_id IS NULL
''')
checks["orphaned line_items"] = int(orphans.iloc[0, 0])

for k, v in checks.items():
    status = "PASS" if v == 0 else "FAIL"
    print(f"[{status}] {k}: {v}")

## Section 6 — Sequence behaviour and the invoice display number

Key properties from the [Sequences reference article](https://www.singlestore.com/blog/guide-to-singlestore-sequences/):

<table>
  <tr><th>Property</th><th>Behaviour</th></tr>
  <tr><td><strong>Unique</strong></td><td>Every generated value is guaranteed unique</td></tr>
  <tr><td><strong>Sequential but not gapless</strong></td><td>Rolled-back transactions leave unused values; small gaps are normal</td></tr>
  <tr><td><strong>Cached per aggregator</strong></td><td>Each aggregator pre-fetches a range (default 100 values); jumps between aggregators are expected</td></tr>
  <tr><td><strong>One sequence per table</strong></td><td>Only one <code>AUTO_INCREMENT AS SEQUENCE</code> column is supported per table</td></tr>
  <tr><td><strong>Not an idempotency key</strong></td><td>The sequence does not deduplicate; use the source event key for that</td></tr>
</table>

**Invoice display number vs. invoice number:**

`CONCAT('INV-', LPAD(order_id, 6, '0'))` is an **invoice display number** — a presentation-layer label derived on the fly from `order_id`. It is unique for the rows in this notebook because `order_id` is unique, but:

* It is application-facing only, not an accounting-controlled or legally gapless invoice number
* If your system requires a formal invoice sequence (e.g. for billing or compliance), create a dedicated `invoices` table with its own `INT AUTO_INCREMENT AS SEQUENCE PRIMARY KEY`
* Treat `INV-000042` as a display label, not a durable external key

In [13]:
# Invoice display number derived in SQL — not stored as a column
run_sql('''
    SELECT t.txn_id,
           CONCAT("TXN-", LPAD(t.txn_id, 5, "0"))        AS display_ref,
           ord.order_id,
           CONCAT("INV-", LPAD(ord.order_id, 6, "0"))     AS invoice_display_number,
           t.customer, t.category, t.bid_cents
    FROM   marketing_txns t
    JOIN   mktx_orders    ord ON ord.txn_id = t.txn_id
    ORDER  BY t.txn_id
    LIMIT  8
''')

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Manual override warning</b></p>
        <p>Manually supplying a value for a sequence column (e.g. <code>INSERT INTO marketing_txns (txn_id, ...) VALUES (99, ...)</code>) can desynchronise the aggregator caches. If you ever do this, run:</p>
        <pre>AGGREGATOR SYNC AUTO_INCREMENT ON marketing_txns ALL;</pre>
        <p>This notebook does <strong>not</strong> perform manual overrides in the main path.</p>
    </div>
</div>

## Section 7 — API and support lookup

This is the query an API endpoint, support workflow, or AI assistant could run to retrieve the full context for a given transaction: the original purchase event, subscriber context, vendor enrichment, generated identifiers, and line items.

In [14]:
run_sql('''
    SELECT
        t.txn_id,
        CONCAT("TXN-", LPAD(t.txn_id, 5, "0"))         AS display_ref,
        ord.order_id,
        CONCAT("INV-", LPAD(ord.order_id, 6, "0"))      AS invoice_display_number,
        t.subscriber_id,
        t.event_ts,
        t.customer                                        AS advertiser,
        t.category,
        t.domain                                          AS vendor_domain,
        t.bid_cents,
        l.line_id,
        l.vendor,
        l.bid_cents                                       AS line_bid_cents
    FROM       marketing_txns  t
    JOIN       mktx_orders     ord ON ord.txn_id = t.txn_id
    LEFT JOIN  txn_line_items  l   ON l.txn_id   = t.txn_id
    ORDER BY   t.txn_id DESC
    LIMIT 5
''')

### Extending the lookup

The query above returns everything needed to explain a transaction to a customer, populate a support ticket, or feed context into an AI assistant. No external credentials are required — all context is stored in the application tables.

If your workspace has AI Functions enabled, you can extend the SELECT with:

```sql
cluster.EMBED_TEXT(CONCAT(t.customer, ' ', t.category)) AS embedding
```

to add semantic embeddings for similarity search. This is optional and does not affect the core schema design demonstrated in this notebook.

In [15]:
# Runnable lookup by transaction ID — no external credentials required.
# Replace the WHERE clause with a specific txn_id to look up a single transaction.
run_sql('''
    SELECT
        t.txn_id,
        CONCAT("INV-", LPAD(ord.order_id, 6, "0"))  AS invoice_display_number,
        t.subscriber_id,
        t.event_ts,
        t.customer                                    AS advertiser,
        t.category,
        t.domain                                      AS vendor_domain,
        t.bid_cents,
        l.line_id,
        l.vendor,
        l.bid_cents                                   AS line_bid_cents
    FROM       marketing_txns  t
    JOIN       mktx_orders     ord ON ord.txn_id  = t.txn_id
    LEFT JOIN  txn_line_items  l   ON l.txn_id    = t.txn_id
    ORDER BY   t.txn_id DESC
    LIMIT 5
''')

## Key takeaways

<table>
  <tr><th>Concept</th><th>What this notebook showed</th></tr>
  <tr><td><code>AUTO_INCREMENT AS SEQUENCE</code></td><td>Database-generated identifiers — unique, sequential, no application coordination needed</td></tr>
  <tr><td>Source event identity</td><td><code>(subscriber_id, event_ts, offer_id)</code> travels with each row for lineage and duplicate detection</td></tr>
  <tr><td>Parent-child linking</td><td><code>LAST_INSERT_ID()</code> returns the sequence value generated for the current session — safe under concurrent writes</td></tr>
  <tr><td>Sequence gaps</td><td>Rolled-back transactions leave gaps; non-contiguous values are normal and expected</td></tr>
  <tr><td>Invoice display number</td><td><code>CONCAT('INV-', LPAD(order_id, 6, '0'))</code> is a presentation-layer label, not an accounting invoice sequence</td></tr>
  <tr><td>Idempotency</td><td>With a sequence PK, composite unique constraints are not supported; guard duplicate events at the application layer</td></tr>
</table>

## Section 8 — Cleanup and rerun guidance

**To rerun the notebook from scratch:** run this cleanup cell, then re-execute from Section 2. All source and application tables will be recreated cleanly.

**The `martech` database is never touched** — only objects created by this notebook are dropped.

**Rerun safety:** The insert in Section 5 truncates the application tables before loading, so re-executing individual insert cells is also safe.

In [16]:
app_tables  = ["txn_line_items", "mktx_orders", "marketing_txns"]
src_tables  = ["rtdm_purchases", "rtdm_offers", "rtdm_vendors"]

if not IS_SHARED:
    run_sql(f"DROP DATABASE IF EXISTS {DB_NAME}")
    print(f"Database {DB_NAME!r} dropped.")
else:
    for t in app_tables + src_tables:
        run_sql(f"DROP TABLE IF EXISTS {t}")
        print(f"Dropped table: {t}")
print("Cleanup complete.")

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>